In [ ]:
import os
os.environ["OMP_NUM_THREADS"] = "2"
os.environ["MKL_NUM_THREADS"] = "2"
os.environ["OPENBLAS_NUM_THREADS"] = "2"
os.environ["NUMEXPR_NUM_THREADS"] = "2"

import torch
torch.set_num_threads(2)
torch.set_num_interop_threads(2)
torch.cuda.set_device(3)
print(f"Using GPU: {torch.cuda.current_device()} "
      f"({torch.cuda.get_device_name(3)})")

In [ ]:
from IPython.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))
%matplotlib notebook
from argparse import ArgumentParser
import yaml
import os
import math
import torch
# from torch import vmap
from torch.func import vmap, grad
from models import FNN2d
from train_utils import Adam

from solver.BlackScholesEq import BlackScholesEq1D
import traceback

import scipy.io
import torch.nn.functional as F
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

import numpy as np
import imageio


from tqdm import tqdm
from train_utils.utils import save_checkpoint, load_checkpoint, update_config, load_config
from train_utils.losses import LpLoss
from train_utils.datasets import DataLoader1D

from importlib import reload

try:
    import wandb
except ImportError:
    wandb = None


# Solver Sanity Check

Verify the reference Crank–Nicolson solver before generating training data. We march backward in **τ = T − t** from the terminal payoff (τ = 0) to today's price (τ = T).

In [ ]:
%matplotlib inline
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Problem parameters
K = 100.0          # strike
r = 0.05           # risk-free rate
sigma = 0.2        # volatility
T = 1.0            # time to maturity
S_min, S_max = 1.0, 200.0
Nx = 256
dtau = 1e-3

bs = BlackScholesEq1D(
    S_min=S_min, S_max=S_max, Nx=Nx,
    r=r, sigma=sigma, dtau=dtau, T=T,
    device=device,
)

# Terminal payoff is the initial condition in tau coordinates
v0 = bs.call_payoff(K)
bc_lo, bc_hi = bs.call_bcs(K)

# Save ~50 snapshots over [0, T]
save_interval = max(1, int(T / dtau / 50))
V = bs.bs_driver(v0, bc_lo, bc_hi, save_interval=save_interval)

S = bs.S_grid.detach().cpu()
tau_list = bs.T_list

print(f'device: {device}')
print(f'solution shape: {tuple(V.shape)}   # (n_snapshots, Nx)')
print(f'first tau: {tau_list[0]:.4f}, last tau: {tau_list[-1]:.4f}')

# Plot a few tau slices
fig, ax = plt.subplots(figsize=(8, 4))
for idx in [0, len(V) // 2, -1]:
    ax.plot(S, V[idx].detach().cpu(), label=fr'$\tau={tau_list[idx]:.3f}$')
ax.set_xlabel('S')
ax.set_ylabel('V(S, τ)')
ax.set_title(f'European call, K={K}, r={r}, σ={sigma}')
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

# Optional: compare final slice to closed-form Black–Scholes at t=0
def bs_call_price(S, K, r, sigma, T_rem):
    """Analytical European call; T_rem = time to maturity."""
    S = torch.as_tensor(S, dtype=torch.float64)
    if T_rem <= 0:
        return torch.clamp(S - K, min=0.0)
    d1 = (torch.log(S / K) + (r + 0.5 * sigma**2) * T_rem) / (sigma * math.sqrt(T_rem))
    d2 = d1 - sigma * math.sqrt(T_rem)
    from torch.special import ndtr
    return S * ndtr(d1) - K * math.exp(-r * T_rem) * ndtr(d2)

V_final = V[-1].detach().cpu()
V_exact = bs_call_price(S, K, r, sigma, T).cpu()
rel_err = torch.norm(V_final - V_exact) / torch.norm(V_exact)
print(f'relative L2 error vs analytical BS at τ=T: {rel_err.item():.4e}')

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(S, V_final, label='CN solver')
ax.plot(S, V_exact, '--', label='analytical BS')
ax.set_xlabel('S')
ax.set_ylabel('V(S, T)')
ax.set_title('Today\'s price: solver vs analytical')
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()